[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/07_difiisr_bicubic_input_dataset.ipynb)


# 07 — DifIISR Bicubic Input Dataset

**Goal:** create, persist, validate, and inspect the exact image-space input produced immediately before the DifIISR autoencoder encoder.

For frozen DifIISR commit `09ca97ea48d481656dd8090e84099c963059ac41`, the path is:

`LR → normalize [-1,1] → reflect-pad to multiple of 64 → PyTorch bicubic ×4 → autoencoder encoder`

This notebook stops immediately before the encoder. No model inference, weights, or GPU are required. It stores exact float32 NPY data, viewable cropped PNGs, a manifest, and reproducibility metadata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
EXPORT_ROOT=DATASET_ROOT/'difiisr_bicubic_input_x4'
EXACT_DIR=EXPORT_ROOT/'exact_padded_float32'
PNG_DIR=EXPORT_ROOT/'cropped_png'
EXACT_DIR.mkdir(parents=True,exist_ok=True); PNG_DIR.mkdir(parents=True,exist_ok=True)
assert LR_DIR.exists(), f'LR directory not found: {LR_DIR}'
print('LR source:',LR_DIR); print('Export root:',EXPORT_ROOT)

In [ ]:
import json,time,numpy as np,pandas as pd,torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
EXTS={'.png','.jpg','.jpeg'}
images=sorted(p for p in LR_DIR.iterdir() if p.suffix.lower() in EXTS)
print('Number of LR images:',len(images))
assert len(images)==2115, f'Expected 2115 validation images, found {len(images)}'

## Exact preprocessing
No clamp is applied after bicubic interpolation: any interpolation overshoot is part of the numeric tensor seen by the encoder.

In [ ]:
def pad64(y):
    h,w=y.shape[-2:]; ph=(64-h%64)%64; pw=(64-w%64)%64
    return F.pad(y,(0,pw,0,ph),mode='reflect'),h,w,ph,pw

def load_and_upsample_exact(path):
    arr=np.asarray(Image.open(path).convert('RGB'),dtype=np.float32)/255.0
    y=torch.from_numpy(arr).permute(2,0,1).unsqueeze(0)
    y=(y-0.5)/0.5
    y_pad,h,w,ph,pw=pad64(y)
    y_up=F.interpolate(y_pad,scale_factor=4,mode='bicubic')
    return y_up,h,w,ph,pw

def to_uint8_rgb(x):
    a=((x[0].float().cpu()+1.0)*0.5*255.0)
    return a.clamp(0,255).permute(1,2,0).numpy().round().astype(np.uint8)

In [ ]:
sample_path=images[0]
y_up,h,w,ph,pw=load_and_upsample_exact(sample_path)
y_crop=y_up[...,:h*4,:w*4]
print('File:',sample_path.name)
print(f'Original LR: {w} x {h}')
print(f'LR padding: right={pw}, bottom={ph}')
print(f'Exact encoder input: {y_up.shape[-1]} x {y_up.shape[-2]}')
print(f'Cropped x4: {y_crop.shape[-1]} x {y_crop.shape[-2]}')
print(f'Float range: [{y_up.min().item():.6f}, {y_up.max().item():.6f}]')
plt.figure(figsize=(10,6)); plt.imshow(to_uint8_rgb(y_crop)); plt.title(sample_path.name); plt.axis('off'); plt.show()

## Full export
Resume-safe: existing NPY + PNG pairs are reused. NPY is authoritative for quantitative SCR work; cropped float data is recovered losslessly by slicing the padded NPY, so no second large float copy is needed.

In [ ]:
rows=[]; start=time.time()
for idx,ip in enumerate(images,1):
    stem=ip.stem; npy_path=EXACT_DIR/f'{stem}.npy'; png_path=PNG_DIR/f'{stem}.png'
    with Image.open(ip) as im: w0,h0=im.size
    ph0=(64-h0%64)%64; pw0=(64-w0%64)%64
    if not (npy_path.exists() and png_path.exists()):
        y_up,h,w,ph,pw=load_and_upsample_exact(ip); y_crop=y_up[...,:h*4,:w*4]
        np.save(npy_path,y_up[0].cpu().numpy().astype(np.float32))
        Image.fromarray(to_uint8_rgb(y_crop)).save(png_path)
        fmin,fmax,fmean,fstd=map(float,[y_up.min(),y_up.max(),y_up.mean(),y_up.std()])
    else:
        a=np.load(npy_path,mmap_mode='r'); fmin,fmax,fmean,fstd=float(a.min()),float(a.max()),float(a.mean()),float(a.std())
    rows.append({'image':ip.name,'stem':stem,'lr_width':w0,'lr_height':h0,'pad_right_lr':pw0,'pad_bottom_lr':ph0,'padded_lr_width':w0+pw0,'padded_lr_height':h0+ph0,'encoder_input_width':(w0+pw0)*4,'encoder_input_height':(h0+ph0)*4,'cropped_width':w0*4,'cropped_height':h0*4,'float_min':fmin,'float_max':fmax,'float_mean':fmean,'float_std':fstd,'exact_npy':str(npy_path),'cropped_png':str(png_path)})
    if idx%100==0 or idx==len(images):
        elapsed=time.time()-start; rate=idx/elapsed if elapsed else 0; eta=(len(images)-idx)/rate/60 if rate else 0
        print(f'{idx:4d}/{len(images)} | {rate:.2f} img/s | ETA {eta:.1f} min',flush=True)
manifest=pd.DataFrame(rows).sort_values('image').reset_index(drop=True)
manifest.to_csv(EXPORT_ROOT/'manifest.csv',index=False)
print('Saved manifest:',EXPORT_ROOT/'manifest.csv')

In [ ]:
config={'dataset':'DifIISR bicubic x4 pre-encoder input','source_lr_dir':str(LR_DIR),'n_images':len(images),'difiisr_commit':'09ca97ea48d481656dd8090e84099c963059ac41','load':"PIL.Image.open(...).convert('RGB') -> np.float32 / 255.0",'normalization':'(x - 0.5) / 0.5','padding':'torch.nn.functional.pad, reflect, right/bottom to multiple of 64','upsampling':'torch.nn.functional.interpolate','scale_factor':4,'mode':'bicubic','align_corners':'not supplied; PyTorch default','exact_storage':'float32 NPY, CxHxW, padded, no clamp after bicubic','visual_storage':'uint8 RGB PNG, cropped to original field-of-view x4','recover_exact_crop':'a[:, :4*lr_height, :4*lr_width] using manifest.csv','purpose':'SCR calibration and analysis on the image-space input seen by the DifIISR autoencoder encoder'}
with open(EXPORT_ROOT/'dataset_config.json','w') as f: json.dump(config,f,indent=2)
print(json.dumps(config,indent=2))

## Dataset validation

In [ ]:
n_npy=len(list(EXACT_DIR.glob('*.npy'))); n_png=len(list(PNG_DIR.glob('*.png')))
print('Expected:',len(images),'NPY:',n_npy,'PNG:',n_png)
assert n_npy==len(images) and n_png==len(images) and len(manifest)==len(images)
for row in manifest.iloc[[0,len(manifest)//2,len(manifest)-1]].itertuples():
    a=np.load(row.exact_npy,mmap_mode='r'); expected=(3,row.encoder_input_height,row.encoder_input_width)
    assert a.shape==expected and a.dtype==np.float32
print('✓ Dataset complete; sample tensor shapes verified; exact data float32')

## Basic analysis of the upsampled dataset

In [ ]:
summary=pd.DataFrame({'metric':['images','unique LR sizes','unique encoder-input sizes','images with bicubic values < -1','images with bicubic values > 1'],'value':[len(manifest),manifest[['lr_width','lr_height']].drop_duplicates().shape[0],manifest[['encoder_input_width','encoder_input_height']].drop_duplicates().shape[0],int((manifest.float_min<-1).sum()),int((manifest.float_max>1).sum())]})
display(summary)
display(manifest.groupby(['lr_width','lr_height']).size().reset_index(name='n_images').sort_values('n_images',ascending=False))

In [ ]:
rng=np.random.default_rng(12345); sel=rng.choice(len(manifest),size=min(6,len(manifest)),replace=False)
for j in sel:
    row=manifest.iloc[j]; img=Image.open(row.cropped_png)
    plt.figure(figsize=(8,5)); plt.imshow(img); plt.title(f"{row.image} | LR {row.lr_width}×{row.lr_height} → Bicubic {row.cropped_width}×{row.cropped_height}"); plt.axis('off'); plt.show()

## Helper for downstream SCR calibration
Use NPY, not PNG, for quantitative SCR. This removes only the padding region and never reruns bicubic interpolation.

In [ ]:
def load_exact_cropped(stem_or_filename):
    stem=Path(stem_or_filename).stem; row=manifest.loc[manifest.stem==stem]
    if len(row)!=1: raise KeyError(f'Expected one manifest row for {stem}, found {len(row)}')
    row=row.iloc[0]; x=np.load(row.exact_npy); x=x[:,:int(row.lr_height)*4,:int(row.lr_width)*4]
    return x,row
x,row=load_exact_cropped(manifest.iloc[0].image)
print('Exact cropped shape:',x.shape,'dtype:',x.dtype,'range:',float(x.min()),float(x.max()))

## Output to share
Persistent Drive folder: `/content/drive/MyDrive/thermal-drone-dataset/difiisr_bicubic_input_x4/`

For SCR calibration, share the whole folder. Existing HR-aligned labels should remain in the same cropped ×4 spatial frame; verify a few GT overlays before the full study.